# Chapter 10 &mdash; The Nine Derivative Rules

**Concept 4 of the Chapter 10 decomposition:** *The Nine Derivative Rules*

Structural recursion over the RE forms &mdash; with concatenation splitting on whether $E_1$ is nullable.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter10/Concept-Nine-Derivative-Rules/Concept-Nine-Derivative-Rules.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_rederiv    import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


One rule per syntactic form:

| $E$ | $E_c$ |
|---|---|
| $\emptyset$ | $\emptyset$ |
| $\varepsilon$ | $\emptyset$ |
| $a$ | $\varepsilon$ if $a=c$, else $\emptyset$ |
| $E_1 + E_2$ | $(E_1)_c + (E_2)_c$ |
| $E_1 \& E_2$ | $(E_1)_c \,\&\, (E_2)_c$ |
| $!E$ | $!(E_c)$ |
| $E^*$ | $E_c\,E^*$ |
| $E_1E_2$, $E_1$ **not** nullable | $(E_1)_c E_2$ |
| $E_1E_2$, $E_1$ nullable | $(E_1)_c E_2 + (E_2)_c$ |

**The concatenation split is the only subtle rule.** If $E_1$ can match
$\varepsilon$, then $c$ might belong to $E_2$ instead &mdash; so both possibilities must
be unioned. Getting that wrong is the classic bug, and it only shows up on inputs
where $E_1$ matches nothing.

## 2. Definitions

### The matcher

In [ ]:
# --- the derivative matcher, in full -------------------------------------
# AST forms produced by re2ast:
#    ('@','@')            epsilon
#    ('str', c)           a single symbol
#    ('+', (E1, E2))      union
#    ('.', (E1, E2))      concatenation
#    ('*', E)             star
#    ('!', E)             negation
#    ('&', (E1, E2))      intersection
EPS   = ('@', '@')
PHI   = ('phi', 'phi')          # the empty language -- not produced by the
                                # parser, but the derivative rules need it

def nullable(E):
    t = E[0]
    if t == '@'  : return True
    if t == 'phi': return False
    if t == 'str': return False
    if t == '+'  : return nullable(E[1][0]) or  nullable(E[1][1])
    if t == '&'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '.'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '*'  : return True
    if t == '!'  : return not nullable(E[1])
    raise ValueError(E)

def dv(c, E):
    t = E[0]
    if t == '@'  : return PHI
    if t == 'phi': return PHI
    if t == 'str': return EPS if E[1] == c else PHI
    if t == '+'  : return ('+', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '&'  : return ('&', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '*'  : return ('.', (dv(c, E[1]), E))
    if t == '!'  : return ('!', dv(c, E[1]))
    if t == '.'  :
        E1, E2 = E[1]
        left = ('.', (dv(c, E1), E2))
        return ('+', (left, dv(c, E2))) if nullable(E1) else left
    raise ValueError(E)

def matches(s, E):
    for ch in s:
        E = dv(ch, E)
    return nullable(E)

def rmatch(restr, s):
    return matches(s, re2ast(restr)[0])

### A deliberately BUGGY concatenation rule, for contrast

In [ ]:
def dv_buggy(c, E):
    t = E[0]
    if t in ('@', 'phi'): return PHI
    if t == 'str': return EPS if E[1] == c else PHI
    if t == '+'  : return ('+', (dv_buggy(c, E[1][0]), dv_buggy(c, E[1][1])))
    if t == '&'  : return ('&', (dv_buggy(c, E[1][0]), dv_buggy(c, E[1][1])))
    if t == '*'  : return ('.', (dv_buggy(c, E[1]), E))
    if t == '!'  : return ('!', dv_buggy(c, E[1]))
    if t == '.'  : return ('.', (dv_buggy(c, E[1][0]), E[1][1]))   # BUG: no split
    raise ValueError(E)

def matches_buggy(s, E):
    for ch in s: E = dv_buggy(ch, E)
    return nullable(E)

<!-- nav-strip -->

---

&larr;&nbsp;[Ch10&nbsp;3.&nbsp;The Matching Algorithm: Steer by Derivatives, Decide by Nullability](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter10/Concept-Matching-Algorithm/Concept-Matching-Algorithm.ipynb) &nbsp;&middot;&nbsp; [**Chapter 10** index](https://github.com/ganeshutah/Jove/blob/master/Chapter10/README.md) &nbsp;&middot;&nbsp; [Ch10&nbsp;5.&nbsp;Rule 6: The Derivative of a Negation, and Why Deferring Works](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter10/Concept-Rule-6-Negation/Concept-Rule-6-Negation.ipynb)&nbsp;&rarr;

---

## 3. Tests

The primitive rules.

In [ ]:
print("dv('0', epsilon)   =", dv('0', EPS),   " -> the empty language")
print("dv('0', phi)       =", dv('0', PHI))
print("dv('0', ('str','0'))=", dv('0', ('str', '0')), " -> epsilon")
print("dv('1', ('str','0'))=", dv('1', ('str', '0')), " -> the empty language")
assert dv('0', ('str', '0')) == EPS and dv('1', ('str', '0')) == PHI

The **star** rule unrolls once and keeps the star.

In [ ]:
E = re2ast("0*")[0]
d = dv('0', E)
print("dv('0', 0*) =", d)
assert d[0] == '.' and d[1][1] == E
print("  -> (dv of the body) . (the original star)")

The **concatenation split**, both branches.

In [ ]:
E1 = re2ast("01")[0]        # 0 is NOT nullable
E2 = re2ast("0*1")[0]       # 0* IS nullable
print("dv('0', 01)  :", dv('0', E1)[0], " (single branch)")
print("dv('1', 0*1) :", dv('1', E2)[0], " (a '+' -- both branches)")
assert dv('0', E1)[0] == '.'
assert dv('1', E2)[0] == '+'

The buggy version gets exactly the nullable-prefix cases wrong.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(8) for p in product('01', repeat=k)]
for restr in ["0*1", "(0+1)*01", "0*0*1"]:
    E = re2ast(restr)[0]
    diff = [s for s in strs if matches(s, E) != matches_buggy(s, E)]
    print("%-14s differs on %3d strings, shortest %r"
          % (restr, len(diff), diff[0] if diff else None))
    assert diff, "the bug must show up when the left operand is nullable"

Where the left operand is never nullable the bug is invisible &mdash; hence 'classic'.

In [ ]:
for restr in ["0", "0+1", "(0+1)"]:
    E = re2ast(restr)[0]
    diff = [s for s in strs if matches(s, E) != matches_buggy(s, E)]
    print("%-8s buggy rule differs on : %s" % (restr, diff))
    assert not diff
print()
print("No concatenation node, so the broken rule is never reached.")
print("But even '01' breaks it: after one symbol the left operand becomes")
print("epsilon, which IS nullable --")
E = re2ast("01")[0]
d = dv('0', E)
print("   dv('0','01') =", d, " left operand nullable?", nullable(d[1][0]))
assert nullable(dv('0', E)[1][0])
print("-- so the split is needed on the very next step.  A test suite of")
print("patterns with no concatenation at all would have passed. That is the trap.")

All nine rules, exercised against the DFA route.

In [ ]:
for restr in ["''", "0", "0+1", "01", "0*", "0*1", "(0+1)*01", "!(0*)", "(0*)&(0+1)*"]:
    E = re2ast(restr)[0]
    try:
        D = min_dfa(nfa2dfa(re2nfa(restr)))
        sig = sorted(D["Sigma"]) or ['0']
        ss = [''.join(p) for k in range(8) for p in product(sig, repeat=k)]
        ok = all(matches(s, E) == accepts_dfa(D, s) for s in ss)
        print("%-14s agrees with the DFA route : %s" % (restr, ok))
        assert ok
    except Exception:
        ok = all(matches(s, E) is not None for s in strs)
        print("%-14s (no RE2NFA support for ! or &) -- matcher runs : %s" % (restr, ok))

## 4. Exercises


1. Write the nine rules on one sheet from memory. Which did you forget?
2. Prove the concatenation split correct from $L(E_c)=\{w : cw\in L(E)\}$.
3. Add a rule for $E^+$ (one or more). Can you derive it from the star rule?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter10/Concept-Nine-Derivative-Rules')